In [1]:
import pandas as pd
import sqlite3
import os

# Load cleaned movie dataset
movies = pd.read_csv("../data/movies_cleaned.csv")

# Create SQLite database
conn = sqlite3.connect("../data/tmdb_movies.db")

# Store movies data in SQLite
movies.to_sql("movies", conn, if_exists="replace", index=False)

print("SQLite database created successfully!")
print("Total movies:", len(movies))

SQLite database created successfully!
Total movies: 2503


In [2]:
# Check the movies table
query = "SELECT COUNT(*) AS total_movies FROM movies"

result = pd.read_sql_query(query, conn)

print("SQL DATABASE CHECK")
display(result)

# Show first 5 movies
query = "SELECT movie_id, title, release_date, revenue FROM movies LIMIT 5"

display(pd.read_sql_query(query, conn))

SQL DATABASE CHECK


,total_movies
0,2503


,movie_id,title,release_date,revenue
0,157336,Interstellar,2014-11-05,7.466067e+08
1,27205,Inception,2010-07-15,8.390306e+08
2,24428,The Avengers,2012-04-25,1.518816e+09
3,155,The Dark Knight,2008-07-16,1.004558e+09
4,19995,Avatar,2009-12-16,2.923706e+09


In [3]:
query = """
SELECT
    title,
    revenue,
    budget,
    vote_average
FROM movies
WHERE revenue > 0
ORDER BY revenue DESC
LIMIT 10;
"""

top_revenue_movies = pd.read_sql_query(query, conn)

display(top_revenue_movies)

,title,revenue,budget,vote_average
0,Avatar,2.923706e+09,237000000.0,7.610
1,Avengers: Endgame,2.799439e+09,356000000.0,8.238
2,Avatar: The Way of Water,2.334485e+09,460000000.0,7.600
3,Titanic,2.264162e+09,200000000.0,7.901
4,Star Wars: The Force Awakens,2.068224e+09,245000000.0,7.247
5,Avengers: Infinity War,2.052415e+09,300000000.0,8.240
6,Spider-Man: No Way Home,1.921207e+09,200000000.0,7.934
7,Zootopia 2,1.868209e+09,150000000.0,7.680
8,Inside Out 2,1.698864e+09,200000000.0,7.543
9,Jurassic World,1.671537e+09,150000000.0,6.700


In [4]:
query = """
SELECT
    title,
    popularity,
    vote_average,
    vote_count
FROM movies
WHERE popularity > 0
ORDER BY popularity DESC
LIMIT 10;
"""

top_popular_movies = pd.read_sql_query(query, conn)

display(top_popular_movies)

,title,popularity,vote_average,vote_count
0,Spider-Man: No Way Home,438.4757,7.934,22429
1,Disclosure Day,394.3346,7.427,2328
2,Obsession,364.8722,8.248,4203
3,Backrooms,258.4743,7.100,2483
4,Spider-Man,180.5936,7.339,20991
5,Spider-Man: Homecoming,169.0857,7.330,23456
6,Project Hail Mary,162.7023,8.659,6750
7,Michael,108.7248,8.696,3846
8,Spider-Man: Far From Home,105.6644,7.395,17240
9,Mortal Kombat II,99.5231,7.906,2260


In [5]:
query = """
SELECT
    ROUND(AVG(budget), 2) AS average_budget,
    ROUND(AVG(revenue), 2) AS average_revenue
FROM movies
WHERE budget > 0 AND revenue > 0;
"""

display(pd.read_sql_query(query, conn))

,average_budget,average_revenue
0,61052096.47,2.259435e+08


In [6]:
query = """
SELECT
    strftime('%Y', release_date) AS release_year,
    COUNT(*) AS total_movies
FROM movies
WHERE release_date IS NOT NULL
  AND release_date != ''
GROUP BY release_year
ORDER BY release_year;
"""

movies_per_year = pd.read_sql_query(query, conn)

display(movies_per_year)

,release_year,total_movies
0,1921,1
1,1922,1
2,1927,1
3,1931,2
4,1936,1
...,...,...
84,2022,65
85,2023,62
86,2024,43
87,2025,26


In [7]:
query = """
SELECT
    title,
    vote_average,
    vote_count,
    popularity
FROM movies
WHERE vote_count >= 100
ORDER BY vote_average DESC, vote_count DESC
LIMIT 10;
"""

top_rated_movies = pd.read_sql_query(query, conn)

display(top_rated_movies)

,title,vote_average,vote_count,popularity
0,The Shawshank Redemption,8.727,30915,76.9826
1,Michael,8.696,3846,108.7248
2,The Godfather,8.686,23255,60.1968
3,Project Hail Mary,8.659,6750,162.7023
4,Schindler's List,8.570,17680,30.3727
5,The Godfather Part II,8.570,14116,35.9794
6,12 Angry Men,8.566,10192,30.7453
7,The Dark Knight,8.534,36255,60.5510
8,Spirited Away,8.533,18619,33.4002
9,Dilwale Dulhania Le Jayenge,8.507,4607,14.8639


In [8]:
query = """
SELECT
    ROUND(AVG(runtime), 2) AS average_runtime,
    MIN(runtime) AS shortest_runtime,
    MAX(runtime) AS longest_runtime
FROM movies
WHERE runtime > 0;
"""

runtime_analysis = pd.read_sql_query(query, conn)

display(runtime_analysis)

,average_runtime,shortest_runtime,longest_runtime
0,114.41,45,242


In [9]:
query = """
SELECT
    original_language,
    COUNT(*) AS movie_count,
    ROUND(AVG(revenue), 2) AS average_revenue,
    SUM(revenue) AS total_revenue
FROM movies
WHERE revenue > 0
GROUP BY original_language
ORDER BY total_revenue DESC
LIMIT 10;
"""

language_analysis = pd.read_sql_query(query, conn)

display(language_analysis)

,original_language,movie_count,average_revenue,total_revenue
0,en,2272,2.322641e+08,5.277041e+11
1,ja,32,1.051991e+08,3.366372e+09
2,fr,24,8.276898e+07,1.986456e+09
3,ko,9,7.312806e+07,6.581525e+08
4,it,18,3.028678e+07,5.451620e+08
5,de,10,3.958231e+07,3.958231e+08
6,zh,2,1.956865e+08,3.913730e+08
7,es,12,3.043453e+07,3.652144e+08
8,cn,5,4.375577e+07,2.187788e+08
9,hi,2,8.500000e+07,1.700000e+08


In [10]:
query = """
SELECT
    status,
    COUNT(*) AS movie_count,
    ROUND(AVG(vote_average), 2) AS average_rating,
    SUM(vote_count) AS total_votes
FROM movies
GROUP BY status
ORDER BY movie_count DESC;
"""

status_analysis = pd.read_sql_query(query, conn)

display(status_analysis)

,status,movie_count,average_rating,total_votes
0,Released,2503,6.94,15503169


In [11]:
query = """
SELECT
    title,
    runtime,
    release_date,
    vote_average
FROM movies
WHERE runtime > 0
ORDER BY runtime DESC
LIMIT 10;
"""

longest_movies = pd.read_sql_query(query, conn)

display(longest_movies)

,title,runtime,release_date,vote_average
0,Zack Snyder's Justice League,242,2021-03-18,8.083
1,Gone with the Wind,238,1939-12-15,7.906
2,Once Upon a Time in America,229,1984-05-23,8.373
3,Lawrence of Arabia,228,1962-12-11,7.999
4,Ben-Hur,212,1959-11-18,7.891
5,The Irishman,209,2019-11-01,7.573
6,Seven Samurai,207,1954-04-26,8.450
7,Killers of the Flower Moon,206,2023-10-18,7.392
8,The Godfather Part II,202,1974-12-20,8.570
9,The Lord of the Rings: The Return of the King,201,2003-12-17,8.503


In [12]:
query = """
SELECT
    title,
    vote_count,
    vote_average,
    popularity
FROM movies
WHERE vote_count > 0
ORDER BY vote_count DESC
LIMIT 10;
"""

most_voted_movies = pd.read_sql_query(query, conn)

display(most_voted_movies)

,title,vote_count,vote_average,popularity
0,Interstellar,40566,8.484,73.0452
1,Inception,39727,8.400,55.8497
2,The Avengers,38993,8.062,72.2515
3,The Dark Knight,36255,8.534,60.5510
4,Avatar,34348,7.610,48.1996
5,Deadpool,33003,7.625,29.3387
6,Fight Club,32496,8.438,53.7525
7,Avengers: Infinity War,32435,8.240,81.4079
8,The Shawshank Redemption,30915,8.727,76.9826
9,Pulp Fiction,30542,8.482,42.0350
